# Site-specific paired weekly sensitivity study: 1–7 March 2025

Four one-factor studies (SP/NSP attenuation, BESS power/energy, minimum EV service, and market participation) are run with both Persistence and Perfect. All outputs are isolated. A separate standard full-horizon oracle uses matched fixed baselines and is not an unconditional upper bound for forecast-based DA offers. No executed notebook copies are generated.


In [ ]:
# Reusable NTPLL weekly sensitivity study. This notebook never runs on opening.
from pathlib import Path
import os, json, hashlib, time, contextlib, traceback, re, gc
import numpy as np
import pandas as pd
ROOT = Path.cwd().resolve()
assert (ROOT / 'upscaledev_imp_rollingMPC.ipynb').exists(), 'Start in UPSCALeDEV_2024'
STUDY_ROOT = ROOT / 'Sensitivity_Results/Rolling_24h/site_week_20250301_07'
for name in ['inputs', 'cases', 'tables', 'figures', 'logs']:
    (STUDY_ROOT / name).mkdir(parents=True, exist_ok=True)
SITE = 'NTPLL'
START_DATE = '2025-03-01'
END_DATE_EXCLUSIVE = '2025-03-08'
SERVICE_LEVELS = [1.0, 0.8, 0.5]
MIP_GAP = 0.01
SOLVER_THREADS_WEEKLY = 2
# Emergency limits abort, never silently accept an unfinished optimization.
SOLVER_EMERGENCY_SECONDS = 300
STUDY_WALL_LIMIT_SECONDS = 4.5 * 3600
MAIN_NOTEBOOK = ROOT / 'upscaledev_imp_rollingMPC.ipynb'
EV_FILE = ROOT / '2025Data/Site_Data_2025/NTPLL_EV_2025_QC.csv'
BTM_FILE = ROOT / '2025Data/Site_Data_2025/NTPLL_BTM_2025_15min_QC.csv'
HISTORY_SOURCE = ROOT / 'Results_Rolling/Site_Cases_2025/monthly_runs/2025-03/inputs/NTPLL/FULL_BTM/Perfect_preperiod_dispatch.csv'
HISTORY_FILE = STUDY_ROOT / 'inputs/common_perfect_preperiod_dispatch.csv'
FINANCIAL_COMPONENTS = ['Total Revenue','WM Revenue','TOU Cost','PD Cost','NCD Cost','EV Revenue']
def sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()
def prepare_inputs():
    hist = pd.read_csv(HISTORY_SOURCE)
    hist['Interval start'] = pd.to_datetime(hist['Interval start'])
    hist = hist[hist['Interval start'] < START_DATE].sort_values('Interval start')
    expected = pd.date_range('2025-02-01', pd.Timestamp(START_DATE)-pd.Timedelta(minutes=15), freq='15min')
    assert pd.DatetimeIndex(hist['Interval start']).equals(expected)
    assert hist['Base [kWh]'].notna().all()
    hist.to_csv(HISTORY_FILE,index=False)
    btm=pd.read_csv(BTM_FILE);btm['Interval start']=pd.to_datetime(btm['Interval start'])
    idx=pd.date_range(START_DATE,pd.Timestamp(END_DATE_EXCLUSIVE)+pd.Timedelta(days=1)-pd.Timedelta(minutes=15),freq='15min')
    selected=btm.set_index('Interval start').reindex(idx)
    assert selected[['p_load_kW','p_PV_kW','p_native_net_kW']].notna().all().all()
    assert (selected.p_native_net_kW-selected.p_load_kW+selected.p_PV_kW).abs().max()<2e-5
    ev=pd.read_csv(EV_FILE,low_memory=False)
    ev['Interval start']=pd.to_datetime(ev['Interval start'])
    assert ev['Interval start'].max()>=idx[-1]
    manifest={'site':SITE,'start':START_DATE,'end_exclusive':END_DATE_EXCLUSIVE,
      'history':'Common Perfect February history; no future dispatch used',
      'initial_SOC':0.5,'terminal_SOC':0.5,'initial_PD_threshold_kW':0.,
      'initial_NCD_threshold_kW':0.,'MIPGap':MIP_GAP,
      'data_sha256':{str(p.relative_to(ROOT)):sha256(p) for p in [EV_FILE,BTM_FILE,HISTORY_SOURCE]},
      'demand_charge':'incremental March charges from zero threshold, not full March bill',
      'retail_horizon':'24h on interior days in this isolated comparison, unlike legacy retail-only shrinking consistency mode',
      'baseline':'endogenous per trajectory except explicitly fixed-baseline oracle benchmark',
      'service':'minimum allowed delivery; upper bound always full requested energy; driver revenue on actual delivery',
      'capacity_economics':'operational revenues only; no BESS capital cost',
      'random_factors':'synthetic known profiles, not measured reserve deployment'}
    (STUDY_ROOT/'inputs/study_manifest.json').write_text(json.dumps(manifest,indent=2))
    # Random profiles have exact daily means 0.2 or 0.5, including the lookahead day.
    # The same synthetic factor is used for SP and NSP to isolate time variation.
    for seed, mean in [(11,.2),(22,.2),(33,.2),(44,.5)]:
        rng=np.random.default_rng(seed)
        vals=np.concatenate([2*mean*rng.permutation(np.linspace(0,1,96)) for _ in range(len(idx)//96)])
        pd.DataFrame({'Interval start':idx,'alpha_SP':vals,'alpha_NSP':vals}).to_csv(
          STUDY_ROOT/f'inputs/contingency_random_mean{int(mean*100):03d}_seed{seed}.csv',index=False)
    oracle_book=json.loads((ROOT/'upscaledev_weekly_oracle.ipynb').read_text())
    oracle_ns={'__name__':'weekly_oracle_helpers'}
    exec(''.join(oracle_book['cells'][1]['source']),oracle_ns)
    fixed=oracle_ns['fixed_preperiod_baseline'](HISTORY_FILE,idx,cutoff=START_DATE)
    fixed.rename_axis('Interval start').reset_index().to_csv(STUDY_ROOT/'inputs/fixed_oracle_baseline.csv',index=False)
    return manifest
def case(case_id, family, **kw):
    out=dict(case_id=case_id,family=family,forecast='Persistence',eta=1.0,
      P_BESS_kW=250.,E_BESS_kWh=332.,alpha_SP=.2,alpha_NSP=.2,
      market_mode='full',baseline_policy='endogenous',random_seed=None)
    out.update(kw);return out
CASES=[
    case('reference','reference'),
    case('factor_000','factor',alpha_SP=0.,alpha_NSP=0.),
    case('factor_050','factor',alpha_SP=.5,alpha_NSP=.5),
    case('factor_100','factor',alpha_SP=1.,alpha_NSP=1.),
    *[case(f'factor_random_{s}','factor',random_seed=s) for s in [11,22,33]],
    case('factor_random50_44','factor',random_seed=44,random_mean=.5,alpha_SP=.5,alpha_NSP=.5),
    case('bess_power2','bess',P_BESS_kW=500.),
    case('bess_energy2','bess',E_BESS_kWh=664.),
    case('bess_both2','bess',P_BESS_kW=500.,E_BESS_kWh=664.),
    case('eta_080','eta',eta=.8),
    case('eta_050','eta',eta=.5),
    case('market_retail','market',market_mode='retail_only'),
    case('market_energy','market',market_mode='energy_only'),
    case('perfect_24h','forecast',forecast='Perfect'),
    case('benchmark_persistence','benchmark',baseline_policy='fixed_preperiod'),
    case('benchmark_perfect','benchmark',forecast='Perfect',baseline_policy='fixed_preperiod'),
]
# Paired Perfect cases reuse every physical/scenario setting and random profile.
PERFECT_SENSITIVITY_CASES=[dict(c,case_id=c['case_id']+'_perfect',forecast='Perfect',paired_case_id=c['case_id'])
    for c in CASES if c['family'] in ['factor','bess','eta','market']]
CASES += PERFECT_SENSITIVITY_CASES
prepare_inputs()
display(pd.DataFrame(CASES))


In [ ]:
# All transformations below are explicit, guarded, and saved in each run manifest.
# The tested physical MPC equations are loaded from the main source notebook.
def replace_once(source, old, new):
    if source.count(old)!=1:
        raise ValueError(f'Expected unique source anchor ({source.count(old)}): {old[:90]}')
    return source.replace(old,new,1)
def transformed_main(source, config):
    source=replace_once(source,'RUN_MAIN_LOOP_DIRECT = False','RUN_MAIN_LOOP_DIRECT = True')
    source=replace_once(source,
      'truncate_at_midnight = (WM_Mode == \'retail_only\') or is_last_run_day',
      'truncate_at_midnight = is_last_run_day')
    source=replace_once(source,
      "if Fc_SessionkWh == 'PerfectSessionkWh' and WM_Mode != 'retail_only':",
      "if Fc_SessionkWh == 'PerfectSessionkWh':")
    source=replace_once(source,
      'result = prob.solve(solver=cp.GUROBI, verbose=False, MIPGap=GUROBI_MIPGAP, Threads=SOLVER_THREADS, Presolve=1, ignore_dpp=True)',
      'result = prob.solve(solver=cp.GUROBI, verbose=False, MIPGap=GUROBI_MIPGAP, Threads=SOLVER_THREADS, Presolve=1, ignore_dpp=True, TimeLimit=SOLVER_EMERGENCY_SECONDS)')
    zero_as='c_RU_DA == 0, c_RD_DA == 0, c_SP_DA == 0, c_NSP_DA == 0, c_RU_RT == 0, c_RD_RT == 0, c_SP_RT == 0, c_NSP_RT == 0'
    if config['market_mode']=='energy_only':
        marker='        # NOTE: Combine all constraints\n'
        source=replace_once(source,marker,'        constraints_WM += ['+zero_as+']\n'+marker)
        marker='            penalty_BESS = cp.Constant(0.0)\n            objective = cp.Minimize(Cost_Opt + penalty_WM + penalty_BESS)'
        source=replace_once(source,marker,'            constraints += ['+zero_as+']\n'+marker)
    # A separately labeled fixed-baseline benchmark is used for the oracle.
    if config['baseline_policy']=='fixed_preperiod':
        source=replace_once(source,
          '        B_EV = Baseline_Opt_i / dt_h  # convert kWh to kW for constraints and cost calculation',
          '        B_EV = fixed_baseline_values(pd.date_range(H_Start_DA, periods=H, freq=interval))\n        Baseline_Opt_i = B_EV * dt_h')
        source=replace_once(source,
          '                B_EV = Baseline_Opt_i / dt_h\n',
          "                B_EV = fixed_baseline_values(Time_table_RT['Interval start'])\n                Baseline_Opt_i = B_EV * dt_h\n")
        source=replace_once(source,
          '            Baseline_96[i] = Baseline_Opt_fc[i].repeat(4).reshape(96, 1) #repeat 4 times of the 24 x 1 series',
          "            Baseline_96[i] = (fixed_baseline_values(pd.date_range(TheDate_Day0, periods=96, freq=interval))*dt_h).reshape(96,1)")
    # Collect achieved DA and RT certificates independently of solver return labels.
    source=replace_once(source,'        status = prob.status\n',
      "        record_weekly_solver(prob, 'DA', H_Start_DA, -1)\n        status = prob.status\n")
    marker='                _record_rt_solver_status('
    lines=source.splitlines(True)
    # RT hook stays in the existing recording function; see run_case below.
    compile(source,'weekly_mpc_transformed','exec')
    return source
def run_case(config, days=None, smoke=False, force=False):
    days=list(days or range(1,8))
    case_id=config['case_id']
    output=STUDY_ROOT/('smoke_cases' if smoke else 'cases')/case_id
    output.mkdir(parents=True,exist_ok=True)
    marker=output/'COMPLETE.json'
    if marker.exists() and not force:
        return json.loads(marker.read_text())
    logfile=STUDY_ROOT/'logs'/f'{"smoke_" if smoke else ""}{case_id}.log'
    started=time.time()
    nb=json.loads(MAIN_NOTEBOOK.read_text())
    sources={i:''.join(nb['cells'][i]['source']) for i in [3,5,7,9,11,13,14]}
    source_hash=hashlib.sha256('\n'.join(sources.values()).encode()).hexdigest()
    ns={'__name__':'__main__','display':lambda *a,**k:None}
    try:
        with logfile.open('w',buffering=1) as stream, contextlib.redirect_stdout(stream), contextlib.redirect_stderr(stream):
            exec(sources[3],ns)
            exec(sources[5],ns)
            if 'SERVICE_LEVEL_MIN' not in ns:
                raise RuntimeError('Main notebook minimum-service implementation not ready')
            fc=config['forecast']
            fc_session=f'{fc}SessionkWh';fc_count=f'{fc}NumbEV';fc_arrival='PerfectatArrival'
            mode='retail_only' if config['market_mode']=='retail_only' else 'full'
            ns.update(EV_DATA_FILE=EV_FILE,BASELINE_DISPATCH_FILE=HISTORY_FILE,
              PASSIVE_DER_ENABLED=True,BTM_BUILDING_FILE=BTM_FILE,BTM_PV_FILE=BTM_FILE,
              BTM_LOAD_SCALE=1.,BTM_PV_SCALE=1.,VERSION_DIR=str(output),
              RUN_MONTHS_CONFIG=[3],RUN_DAYS_CONFIG=days,RUN_MODES=[mode],
              WM_Mode=mode,Enable_WM=(mode=='full'),
              Fc_SessionkWh=fc_session,Fc_NumbEV=fc_count,Fc_AtArrival=fc_arrival,
              RUN_CASE1=False,Cases=['Base'],DA_CASE_IDX=1,
              SERVICE_LEVEL_MIN=float(config['eta']),
              P_BESS_max=float(config['P_BESS_kW']),C_BESS=float(config['E_BESS_kWh']),
              AS_ACTIVATION_MODE='caiso_regulation',
              AS_ACTIVATION_CONSTANTS={'RU':.7,'RD':.7,'SP':config['alpha_SP'],'NSP':config['alpha_NSP']},
              AS_ACTIVATION_SCALE={'RU':1.,'RD':1.,'SP':1.,'NSP':1.},
              AS_CONTINGENCY_ACTIVATION_FILE=None,
              SOLVER_THREADS=SOLVER_THREADS_WEEKLY,GUROBI_MIPGAP=MIP_GAP,
              SOLVER_EMERGENCY_SECONDS=SOLVER_EMERGENCY_SECONDS,
              RT_SOLVER_EMERGENCY_TIME_LIMIT=SOLVER_EMERGENCY_SECONDS,
              PLOT_DAILY_6PANEL_DATES=[f'202503{d:02d}' for d in days if d in [1,7]],
              ANALYSIS_DAYS_CONFIG=[f'202503{d:02d}' for d in days],
              CLEAR_IMPLEMENTATION_AT_RUN_START=True,SENSITIVITY_ENABLED=False,
              SAVE_FINAL_FINANCIAL_FIGURES=False,RT_SOLVER_STATUS_COUNTS={},RT_SOLVER_LIMIT_EVENTS=[])
            if config.get('random_seed') is not None:
                ns['AS_CONTINGENCY_ACTIVATION_FILE']=STUDY_ROOT/f"inputs/contingency_random_mean{int(config.get('random_mean',.2)*100):03d}_seed{config['random_seed']}.csv"
            if config['baseline_policy']=='fixed_preperiod':
                fixed=pd.read_csv(STUDY_ROOT/'inputs/fixed_oracle_baseline.csv',parse_dates=['Interval start']).set_index('Interval start')['baseline_kW']
                def fixed_baseline_values(index):
                    arr=fixed.reindex(pd.DatetimeIndex(index)).to_numpy(float)
                    if not np.isfinite(arr).all(): raise ValueError('Fixed baseline coverage missing')
                    return arr
                ns['fixed_baseline_values']=fixed_baseline_values
            solver_rows=[]
            def record_weekly_solver(problem,stage,day,step):
                extra=problem.solver_stats.extra_stats
                row={'stage':stage,'date':str(day),'step':int(step),'status':str(problem.status)}
                for field in ['Status','Runtime','MIPGap','ObjVal','ObjBound','SolCount']:
                    try: row[field]=float(getattr(extra,field))
                    except Exception: row[field]=None
                solver_rows.append(row)
            ns['record_weekly_solver']=record_weekly_solver
            original_record=ns['_record_rt_solver_status']
            def recorded(problem,date_label,interval_idx,case_label):
                original_record(problem,date_label,interval_idx,case_label)
                record_weekly_solver(problem,'RT',date_label,interval_idx)
            ns['_record_rt_solver_status']=recorded
            exec(sources[7],ns)
            exec(sources[9],ns)
            source=transformed_main(sources[11],config)
            exec(compile(source,'weekly_main','exec'),ns)
            exec(sources[13],ns)
            exec(sources[14],ns)
            pd.DataFrame(solver_rows).to_csv(output/'solver_certificates.csv',index=False)
            manifest={**config,'site':SITE,'days':len(days),'dates':[f'2025-03-{d:02d}' for d in days],
              'elapsed_s':time.time()-started,'source_sha256':source_hash,
              'transformed_source_sha256':hashlib.sha256(source.encode()).hexdigest(),
              'initial_SOC':.5,'terminal_SOC':.5,'MIPGap':MIP_GAP,'mode_in_files':mode,
              'forecast_key':f'2025_{fc_session}_{fc_count}_{fc_arrival}',
              'output_path':str(output.relative_to(ROOT))}
            checks=validate_case(output,manifest)
            manifest['validation']=checks
            (output/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
            if not checks['PASS']:
                raise AssertionError(checks)
            marker.write_text(json.dumps(manifest,indent=2))
            (output/'FAILED.json').unlink(missing_ok=True)
        print(f"PASS {case_id}: {time.time()-started:.1f}s ({len(days)} days)",flush=True)
        return manifest
    except BaseException as exc:
        with logfile.open('a') as stream: traceback.print_exc(file=stream)
        (output/'FAILED.json').write_text(json.dumps({'case_id':case_id,'error':repr(exc),'elapsed_s':time.time()-started},indent=2))
        print(f'FAILED {case_id}: {exc!r}; see {logfile}',flush=True)
        raise
    finally:
        import matplotlib.pyplot as plt
        plt.close('all');ns.clear();gc.collect()
def load_case_tables(output,manifest):
    key=manifest['forecast_key'];mode=manifest['mode_in_files']
    daily=pd.read_csv(output/'Plots/Cost'/key/'daily_financial_detail.csv')
    label='Retail only' if mode=='retail_only' else 'Both'
    daily=daily[daily.Case.eq(label)].copy()
    traces=pd.concat([pd.read_csv(output/'Validation_Traces'/f'{key}_{mode}'/f"rolling_validation_trace_{d.replace('-','')}.csv")
        for d in manifest['dates']],ignore_index=True)
    return daily,traces
def validate_case(output,manifest):
    daily,tr=load_case_tables(output,manifest)
    certificates=pd.read_csv(output/'solver_certificates.csv')
    ascols=[f'c_{p}_actual_kW' for p in ['RU','RD','SP','NSP']]
    expected=pd.date_range(manifest['dates'][0],periods=96*manifest['days'],freq='15min')
    accounting=(daily['Total Revenue']-daily[FINANCIAL_COMPONENTS[1:]].sum(axis=1)).abs().max()
    ev_revenue_error=abs(daily['EV Revenue'].sum()-.4*.25*tr['p_EV_kW'].sum())
    prev=np.r_[.5,tr.SOC.to_numpy()[:-1]];power=tr.p_BESS_kW.to_numpy()
    soc_expected=prev+.25/manifest['E_BESS_kWh']*(np.maximum(power,0)*np.sqrt(.9)+np.minimum(power,0)/np.sqrt(.9))
    checks={
      'date_coverage':pd.DatetimeIndex(pd.to_datetime(tr['Interval start'])).equals(expected),
      'days':len(daily),'expected_days':manifest['days'],
      'meter_error_kW':float(tr.meter_balance_residual_kW.abs().max()),
      'meter_identity_kW':float((tr.p_GI_kW-tr.p_EV_kW-tr.p_BESS_kW-tr.p_load_kW+tr.p_PV_kW).abs().max()),
      'energy_position_error_kW':float((tr.p_actual_kW-tr.p_DA_kW-tr.p_RT_deviation_kW).abs().max()),
      'AS_min_kW':float(tr[ascols].min().min()),
      'AS_max_abs_kW':float(tr[[f'c_{p}_{stage}_kW' for p in ['RU','RD','SP','NSP'] for stage in ['DA','RT']]].abs().max().max()),
      'passive_WM_max_kW':float(tr.passive_WM_position_kW.abs().max()),
      'soc_error':float(np.max(np.abs(tr.SOC.to_numpy()-soc_expected))),
      'soc_min':float(tr.SOC.min()),'soc_max':float(tr.SOC.max()),
      'terminal_soc':float(tr.SOC.iloc[-1]),
      'financial_rounding_error_USD':float(accounting),
      'EV_revenue_executed_error_USD':float(ev_revenue_error),
      'EV_delivered_kWh':float(.25*tr.p_EV_kW.sum()),
      'min_up_slack_kW':float(tr.actual_up_capability_slack_kW.min()),
      'min_down_slack_kW':float(tr.actual_down_capability_slack_kW.min()),
      'solver_count':len(certificates),'solver_nonoptimal':int((certificates['status']!='optimal').sum()),
      'time_limit_count':int((certificates.Status==9).sum()),
      'max_gap':float(certificates.MIPGap.fillna(0).max()),
      'finite_trace':bool(np.isfinite(tr.select_dtypes(include='number').to_numpy()).all()),
    }
    checks['PASS']=bool(checks['date_coverage'] and checks['days']==checks['expected_days']
      and checks['meter_error_kW']<1e-4 and checks['meter_identity_kW']<1e-4
      and checks['energy_position_error_kW']<1e-4 and checks['AS_min_kW']>=-1e-4
      and checks['passive_WM_max_kW']<1e-7 and checks['soc_error']<1e-5
      and checks['soc_min']>=.05-1e-5 and checks['soc_max']<=.95+1e-5
      and abs(checks['terminal_soc']-.5)<1e-5
      and checks['financial_rounding_error_USD']<=.031
      and checks['EV_revenue_executed_error_USD']<=.0051*manifest['days']+1e-4
      and checks['min_up_slack_kW']>=-1e-3 and checks['min_down_slack_kW']>=-1e-3
      and checks['solver_nonoptimal']==0 and checks['time_limit_count']==0
      and checks['max_gap']<=.010001 and checks['finite_trace'])
    if manifest['market_mode']=='retail_only':
        checks['PASS'] &= bool(abs(daily['WM Revenue'].sum())<1e-6 and checks['AS_max_abs_kW']<1e-5 and tr[['p_DA_kW','p_RT_deviation_kW']].abs().max().max()<1e-5)
    elif manifest['market_mode']=='energy_only':
        checks['PASS'] &= bool(checks['AS_max_abs_kW']<1e-5)
    return checks


In [ ]:
# Aggregate only validated COMPLETE cases; incomplete/failed cases are reported separately.
def collect_weekly_results():
    weekly=[];daily_rows=[];interval_rows=[];validation=[];sessions=[]
    for marker in sorted((STUDY_ROOT/'cases').glob('*/COMPLETE.json')):
        m=json.loads(marker.read_text());out=marker.parent
        d,t=load_case_tables(out,m)
        sums=d.select_dtypes(include='number').sum().to_dict()
        # Preserve all recorded revenue-decomposition columns, not just total.
        w={k:v for k,v in m.items() if k not in ['validation','dates']}
        w.update({k:float(d[k].sum()) for k in d.columns if k not in ['Date','Case'] and pd.api.types.is_numeric_dtype(d[k])})
        w.update(m['validation'])
        weekly.append(w)
        for frame in [d,t]:
            frame['case_id']=m['case_id'];frame['family']=m['family'];frame['forecast']=m['forecast']
            frame['eta']=m['eta'];frame['baseline_policy']=m['baseline_policy']
        daily_rows.append(d);interval_rows.append(t)
        validation.append({'case_id':m['case_id'],**m['validation']})
        for path in sorted((out/'Dispatch').glob('*_service_validation.csv')):
            s=pd.read_csv(path);s['case_id']=m['case_id'];s['eta']=m['eta'];s['Date']=path.name[:8];sessions.append(s)
    if weekly:
        w=pd.DataFrame(weekly)
        if 'reference' in w.case_id.values:
            references={f:float(w.set_index('case_id').loc[c,'Total Revenue']) for f,c in [('Persistence','reference'),('Perfect','perfect_24h')] if c in w.case_id.values}
            w['Net revenue improvement vs reference [USD]']=w['Total Revenue']-w.forecast.map(references)
            w.loc[w.baseline_policy.ne('endogenous'),'Net revenue improvement vs reference [USD]']=np.nan
        w['scenario_key']=w.case_id.map(lambda c:'reference' if c in ['reference','perfect_24h'] else c.removesuffix('_perfect') if c.endswith('_perfect') and c!='benchmark_perfect' else c)
        w.to_csv(STUDY_ROOT/'tables/weekly_financial_comparison.csv',index=False)
        pd.concat(daily_rows,ignore_index=True).to_csv(STUDY_ROOT/'tables/daily_financial_comparison.csv',index=False)
        pd.concat(interval_rows,ignore_index=True).to_csv(STUDY_ROOT/'tables/interval_results.csv',index=False)
        pd.DataFrame(validation).to_csv(STUDY_ROOT/'tables/validation_summary.csv',index=False)
        if sessions: pd.concat(sessions,ignore_index=True).to_csv(STUDY_ROOT/'tables/session_service_levels.csv',index=False)
        return w
    return pd.DataFrame()
def run_weekly_batch(selected_cases=None, force=False):
    prepare_inputs()
    selected_cases=selected_cases or CASES
    begun=time.monotonic();status=[]
    for c in selected_cases:
        if time.monotonic()-begun>STUDY_WALL_LIMIT_SECONDS:
            status.append({'case_id':c['case_id'],'status':'NOT_STARTED_WALL_BUDGET'})
            continue
        try:
            result=run_case(c,force=force)
            status.append({'case_id':c['case_id'],'status':'PASS','elapsed_s':result['elapsed_s']})
        except BaseException as exc:
            status.append({'case_id':c['case_id'],'status':'FAILED','error':repr(exc)})
        (STUDY_ROOT/'batch_status.json').write_text(json.dumps(status,indent=2))
        collect_weekly_results()
    return pd.DataFrame(status)
# Deliberately OFF by default. Set env RUN_WEEKLY_SENSITIVITY=1 to run all cases,
# or call run_case(CASES[0], days=[1], smoke=True) for the one-day gate.
if os.environ.get('RUN_WEEKLY_SENSITIVITY','0')=='1':
    display(run_weekly_batch())
else:
    print('Weekly batch is OFF. Review CASES, then run the one-day gate before the batch.')


In [ ]:
# Independent post-run audit. No optimization or case output is modified.
def audit_completed_weekly_cases(study_root=None):
    root = Path(study_root) if study_root else STUDY_ROOT
    records, daily_details, service_details = [], [], []
    # Parameters below are extracted from the source configuration, not inferred from results.
    import ast
    source_nb = json.loads(MAIN_NOTEBOOK.read_text())
    parameter_source = ''.join(source_nb['cells'][5]['source'])
    literals = {}
    for node in ast.parse(parameter_source).body:
        if isinstance(node, ast.Assign):
            try: value = ast.literal_eval(node.value)
            except (ValueError, TypeError): continue
            for target in node.targets:
                if isinstance(target, ast.Name): literals[target.id] = value
    soc_min, soc_max = float(literals['SOC_BESS_min']), float(literals['SOC_BESS_max'])
    fixed_path = root/'inputs/fixed_oracle_baseline.csv'
    fixed = (pd.read_csv(fixed_path, parse_dates=['Interval start']).set_index('Interval start')['baseline_kW']
             if fixed_path.exists() else None)
    data = pd.read_csv(EV_FILE, low_memory=False)
    for col in ['Interval start','Interval end','Session start','Session end']:
        data[col] = pd.to_datetime(data[col], errors='raise')
    # Independently reconstruct the exact interval-filtered, capacity-capped requests.
    data = data[(data['Interval start'] >= data['Session start']) & (data['Interval end'] <= data['Session end'])].copy()
    if data.duplicated(['10-digit UID','Interval start']).any():
        raise ValueError('Duplicate EV session interval in audit input')
    data['_date'] = data['Interval start'].dt.strftime('%Y-%m-%d')
    requests = {}
    for (date, uid), group in data.groupby(['_date','10-digit UID']):
        cap = 4.16 if ((group['Interval kWh'] > 1.664) | (group['Interval max demand kW'] > 6.656)).any() else 1.664
        requests[(date, str(uid))] = min(float(group['Interval kWh'].sum()), cap*len(group))
    for marker in sorted((root/'cases').glob('*/COMPLETE.json')):
        manifest = json.loads(marker.read_text()); out = marker.parent
        financial, tr = load_case_tables(out, manifest)
        idx = pd.DatetimeIndex(pd.to_datetime(tr['Interval start']))
        da_cols = ['p_DA_kW']+[f'c_{p}_DA_kW' for p in ['RU','RD','SP','NSP']]
        hourly_variation = float(tr.groupby(idx.floor('h'))[da_cols].agg(lambda x: x.max()-x.min()).max().max())
        allowance = 2*float(manifest['E_BESS_kWh'])*(soc_max-soc_min)
        throughput = (.25*tr.p_BESS_kW.abs()).groupby(idx.normalize()).sum()
        ev_daily = (.25*tr.p_EV_kW).groupby(idx.strftime('%Y-%m-%d')).sum()
        requested_daily = {date: sum(v for (d,uid),v in requests.items() if d==date) for date in manifest['dates']}
        fullservice_error = max(abs(float(ev_daily.get(date,0))-requested_daily[date]) for date in manifest['dates'])
        row = {'case_id':manifest['case_id'],'eta':manifest['eta'],'days':manifest['days'],
            'hourly_DA_max_variation_kW':hourly_variation,
            'daily_BESS_throughput_allowance_kWh':allowance,
            'daily_BESS_throughput_max_kWh':float(throughput.max()),
            'throughput_excess_kWh':max(0.,float(throughput.max())-allowance),
            'full_service_daily_energy_error_kWh':fullservice_error if manifest['eta']==1.0 else None,
            'fixed_baseline_max_error_kW':None,
            'eta_session_count':0,'eta_session_min_shortfall_kWh':0.,'eta_session_upper_excess_kWh':0.,
            'eta_session_input_target_error_kWh':0.,'eta_session_energy_trace_error_kWh':0.,
            'eta_session_date_coverage':True,'eta_session_id_coverage':True}
        for date in manifest['dates']:
            daily_details.append({'case_id':manifest['case_id'],'Date':date,'eta':manifest['eta'],
                'BESS_throughput_kWh':float(throughput.loc[pd.Timestamp(date)]),
                'BESS_throughput_allowance_kWh':allowance,'EV_requested_kWh':requested_daily[date],
                'EV_delivered_kWh':float(ev_daily.get(date,0))})
        if manifest['baseline_policy']=='fixed_preperiod':
            if fixed is None: raise ValueError('Fixed benchmark baseline file missing')
            expected = fixed.reindex(idx).to_numpy(float)
            row['fixed_baseline_max_error_kW'] = float(np.max(np.abs(tr.baseline_kW.to_numpy()-expected))) if np.isfinite(expected).all() else float('inf')
        if manifest['eta'] < 1.0:
            for date in manifest['dates']:
                file = out/'Dispatch'/f"{date.replace('-','')}_service_validation.csv"
                if not file.exists():
                    row['eta_session_date_coverage'] = False
                    continue
                s = pd.read_csv(file, dtype={'session':str})
                actual_ids = set(s.session)
                expected_ids = {uid for (d,uid),v in requests.items() if d==date}
                row['eta_session_id_coverage'] &= actual_ids==expected_ids and not s.session.duplicated().any()
                min_short = np.maximum(manifest['eta']*s.requested_kWh-s.delivered_kWh,0)
                upper_excess = np.maximum(s.delivered_kWh-s.requested_kWh,0)
                input_diff = [abs(float(r.requested_kWh)-requests.get((date,str(r.session)),float('inf'))) for r in s.itertuples()]
                row['eta_session_count'] += len(s)
                row['eta_session_min_shortfall_kWh'] = max(row['eta_session_min_shortfall_kWh'],float(min_short.max()) if len(s) else 0.)
                row['eta_session_upper_excess_kWh'] = max(row['eta_session_upper_excess_kWh'],float(upper_excess.max()) if len(s) else 0.)
                row['eta_session_input_target_error_kWh'] = max(row['eta_session_input_target_error_kWh'],max(input_diff,default=0.))
                row['eta_session_energy_trace_error_kWh'] = max(row['eta_session_energy_trace_error_kWh'],abs(float(s.delivered_kWh.sum())-float(ev_daily.get(date,0))))
                s['case_id']=manifest['case_id'];s['Date']=date
                s['independent_min_shortfall_kWh']=min_short;s['independent_upper_excess_kWh']=upper_excess
                service_details.append(s)
        row['PASS'] = bool(hourly_variation<1e-5 and row['throughput_excess_kWh']<1e-3
            and (manifest['eta']!=1.0 or fullservice_error<1e-3)
            and (row['fixed_baseline_max_error_kW'] is None or row['fixed_baseline_max_error_kW']<1e-6)
            and row['eta_session_date_coverage'] and row['eta_session_id_coverage']
            and row['eta_session_min_shortfall_kWh']<1e-3 and row['eta_session_upper_excess_kWh']<1e-3
            and row['eta_session_input_target_error_kWh']<1e-6 and row['eta_session_energy_trace_error_kWh']<1e-4)
        records.append(row)
    result = pd.DataFrame(records)
    (root/'tables').mkdir(parents=True,exist_ok=True)
    result.to_csv(root/'tables/extended_validation.csv',index=False)
    pd.DataFrame(daily_details).to_csv(root/'tables/extended_daily_physical_validation.csv',index=False)
    if service_details:
        pd.concat(service_details,ignore_index=True).to_csv(root/'tables/extended_session_service_validation.csv',index=False)
    return result
# Run manually after a batch: display(audit_completed_weekly_cases())


In [ ]:
# Rebuild product-level price / obligation / settlement tables from executed traces.
# This cell is independent of the optimizer and safe to rerun after new cases complete.
def export_weekly_market_audits(root=None):
    from pathlib import Path
    import json
    import numpy as np
    import pandas as pd
    project=Path('/Users/admin/Desktop/EV_program/Total Transfer/PowerFlex_Code/UPSCALeDEV_2024')
    study=Path(root) if root is not None else project/'Sensitivity_Results/Rolling_24h/site_week_20250301_07'
    tables=study/'tables';tables.mkdir(parents=True,exist_ok=True)
    dt=.25
    products={'RU':'RegUp','RD':'RegDown','SP':'Spin','NSP':'NonSpin'}
    def model_clock(series):
        text=series.astype(str).str.strip()
        aware=text.str.contains(r'(?:Z|[+-]\d{2}:\d{2})$',regex=True)
        if aware.all():return pd.to_datetime(text,utc=True).dt.tz_convert('America/Los_Angeles').dt.tz_localize(None)
        if not aware.any():return pd.to_datetime(text,errors='raise')
        raise ValueError('Mixed timezone-aware and naive AS timestamps')
    def as_prices(path):
        frame=pd.read_csv(path);frame['datetime']=model_clock(frame['datetime'])
        if frame['datetime'].duplicated().any():raise ValueError(f'Duplicate AS times: {path}')
        return frame.set_index('datetime').sort_index()
    da=as_prices(project/'2025Data/AS_DAM/AS_price_2025_clear.csv')
    rt=as_prices(project/'2025Data/AS_RTM/AS_price_2025_clear.csv')
    regulation=pd.read_csv(project/'2025Data/AS_Call_Rates/caiso_regulation_attenuation_hourly_2025.csv')
    regulation=regulation.set_index(['quarter','trade_hour'])
    combined=[];daily_all=[];weekly_all=[];checks=[]
    power_columns=['DA quantity (kW)','RT signed adjustment (kW)','Final physical position (kW)','RT buy-back (kW)','RT add-on (kW)','Signed energy obligation (kW)']
    revenue_columns=['DA energy revenue ($)','RT energy revenue ($)','DA capacity revenue ($)','RT capacity revenue ($)','Total product revenue ($)']
    price_columns=['DA LMP ($/kWh)','RT LMP ($/kWh)','DA AS capacity price ($/kWh)','RT AS capacity price ($/kWh)','TOU price ($/kWh)']
    def summarize(frame,keys):
        grouped=frame.groupby(keys,dropna=False,sort=False)
        summary=grouped[revenue_columns].sum()
        for column in power_columns:
            summary[column.replace('(kW)','mean (kW)')]=grouped[column].mean()
            summary[column.replace('(kW)','integral (kWh or kW-h)')]=dt*grouped[column].sum(min_count=1)
        for column in price_columns:
            summary[column.replace('price','mean price').replace('LMP','mean LMP')]=grouped[column].mean()
        summary['Mean attenuation factor']=grouped['Attenuation factor'].mean()
        summary['Intervals']=grouped.size()
        return summary.reset_index()
    for complete in sorted((study/'cases').glob('*/COMPLETE.json')):
        manifest=json.loads(complete.read_text());out=complete.parent
        if not manifest.get('validation',{}).get('PASS',False):continue
        key=manifest['forecast_key'];mode=manifest['mode_in_files']
        traces=[]
        for day in manifest['dates']:
            path=out/'Validation_Traces'/f'{key}_{mode}'/f"rolling_validation_trace_{day.replace('-','')}.csv"
            traces.append(pd.read_csv(path))
        trace=pd.concat(traces,ignore_index=True);idx=pd.DatetimeIndex(pd.to_datetime(trace['Interval start']))
        if idx.duplicated().any():raise ValueError(f'Duplicate trace intervals {out}')
        dap=da.reindex(idx.floor('h'));rtp=rt.reindex(idx)
        if dap[list(products.values())].isna().any().any() or rtp[list(products.values())].isna().any().any():raise ValueError(f'Missing AS prices {out}')
        regkey=pd.MultiIndex.from_tuples([(f'{t.year}Q{t.quarter}',t.hour+1) for t in idx],names=['quarter','trade_hour'])
        reg=regulation.reindex(regkey)
        alpha={'RU':reg.alpha_RU.to_numpy(float),'RD':reg.alpha_RD.to_numpy(float),'SP':np.full(len(idx),float(manifest['alpha_SP'])),'NSP':np.full(len(idx),float(manifest['alpha_NSP']))}
        seed=manifest.get('random_seed')
        if seed is not None and pd.notna(seed):
            random=pd.read_csv(study/f"inputs/contingency_random_mean{int(float(manifest.get('random_mean',.2))*100):03d}_seed{int(seed)}.csv");random['Interval start']=pd.to_datetime(random['Interval start']);random=random.set_index('Interval start').reindex(idx)
            alpha['SP']=random.alpha_SP.to_numpy(float);alpha['NSP']=random.alpha_NSP.to_numpy(float)
        if any(not np.isfinite(a).all() for a in alpha.values()):raise ValueError(f'Missing factor data {out}')
        case_parts=[]
        for product in ['Energy',*products]:
            energy=product=='Energy'
            qda=trace['p_DA_kW'].to_numpy(float) if energy else trace[f'c_{product}_DA_kW'].to_numpy(float)
            qrt=trace['p_RT_deviation_kW'].to_numpy(float) if energy else trace[f'c_{product}_RT_kW'].to_numpy(float)
            a=np.ones(len(idx)) if energy else alpha[product]
            direction=-1 if product=='RD' else 1
            pda=np.zeros(len(idx)) if energy else dap[products[product]].to_numpy(float)*.001
            prt=np.zeros(len(idx)) if energy else rtp[products[product]].to_numpy(float)*.001/dt
            lmpda=trace['LMP_DA_$/kWh'].to_numpy(float);lmprt=trace['LMP_RT_$/kWh'].to_numpy(float)
            eda=dt*qda*(lmpda if energy else direction*a*lmprt)
            ert=dt*qrt*lmprt*(1 if energy else direction*a)
            cda=dt*pda*qda;crt=dt*prt*qrt
            frame=pd.DataFrame({'case_id':manifest['case_id'],'family':manifest['family'],'forecast':manifest['forecast'],'market_mode':manifest['market_mode'],'baseline_policy':manifest['baseline_policy'],'Interval start':idx,'Date':idx.strftime('%Y-%m-%d'),'Product':product,'DA quantity (kW)':qda,'RT signed adjustment (kW)':qrt,'Final physical position (kW)':qda+qrt,'RT buy-back (kW)':np.maximum(-qrt,0),'RT add-on (kW)':np.maximum(qrt,0),'Attenuation factor':a,'Signed energy obligation (kW)':direction*a*(qda+qrt),'DA LMP ($/kWh)':lmpda,'RT LMP ($/kWh)':lmprt,'DA AS capacity price ($/kWh)':pda,'RT AS capacity price ($/kWh)':prt,'TOU price ($/kWh)':trace['TOU_$/kWh'].to_numpy(float),'DA energy revenue ($)':eda,'RT energy revenue ($)':ert,'DA capacity revenue ($)':cda,'RT capacity revenue ($)':crt,'Total product revenue ($)':eda+ert+cda+crt})
            case_parts.append(frame)
        detail=pd.concat(case_parts,ignore_index=True)
        # Total obligation is energy DA+RT plus signed attenuation times actual AS.
        total=detail.groupby('Interval start',sort=True)[power_columns+revenue_columns].sum().reset_index()
        for col in ['case_id','family','forecast','market_mode','baseline_policy']:total[col]=manifest[col]
        total['Date']=pd.to_datetime(total['Interval start']).dt.strftime('%Y-%m-%d');total['Product']='Total';total['Attenuation factor']=np.nan
        for col in price_columns:total[col]=np.nan
        # Sums of capacities across unlike products are not a physical net position.
        total[['DA quantity (kW)','RT signed adjustment (kW)','Final physical position (kW)']]=np.nan
        detail=pd.concat([detail,total],ignore_index=True).sort_values(['Interval start','Product'])
        daily=summarize(detail,['case_id','family','forecast','market_mode','baseline_policy','Date','Product'])
        weekly=summarize(detail,['case_id','family','forecast','market_mode','baseline_policy','Product'])
        expected=pd.read_csv(out/'Plots/Cost'/key/'daily_financial_detail.csv')
        expected=expected[expected.Case.eq('Retail only' if mode=='retail_only' else 'Both')].copy();expected['Date']=pd.to_datetime(expected.Date).dt.strftime('%Y-%m-%d')
        actual=daily[daily.Product.eq('Total')][['Date','Total product revenue ($)']]
        check=actual.merge(expected[['Date','WM Revenue']],on='Date',how='outer',validate='one_to_one')
        check['case_id']=manifest['case_id'];check['WM revenue residual ($)']=check['Total product revenue ($)']-check['WM Revenue'];check['PASS']=check['WM revenue residual ($)'].abs()<=.0051
        if not check['PASS'].all():raise ValueError(f'Product settlement does not reconcile: {check.to_dict("records")}')
        audit_dir=out/'market_audit';audit_dir.mkdir(exist_ok=True)
        detail.to_csv(audit_dir/'market_interval_with_prices.csv',index=False,float_format='%.10f')
        daily.to_csv(audit_dir/'market_daily_with_prices.csv',index=False,float_format='%.10f')
        weekly.to_csv(audit_dir/'market_weekly_with_prices.csv',index=False,float_format='%.10f')
        combined.append(detail);daily_all.append(daily);weekly_all.append(weekly);checks.append(check)
    if not combined:raise ValueError('No completed validated cases available')
    outputs={'market_interval_with_prices.csv':combined,'market_daily_with_prices.csv':daily_all,'market_weekly_with_prices.csv':weekly_all,'market_revenue_reconciliation.csv':checks}
    for name,frames in outputs.items():pd.concat(frames,ignore_index=True).to_csv(tables/name,index=False,float_format='%.10f')
    print(f'Market audit exported for {len(combined)} completed cases; all daily product revenues reconcile within $0.0051.')
    return pd.concat(checks,ignore_index=True)

import os as _market_audit_os
if _market_audit_os.environ.get('RUN_WEEKLY_MARKET_AUDIT','0')=='1':
    export_weekly_market_audits()


In [ ]:
# Finalization is explicit: no simulation or manuscript edit is performed here.
def finalize_weekly_study():
    w=collect_weekly_results()
    missing=set(c['case_id'] for c in CASES)-set(w.case_id)
    if missing: raise RuntimeError('Incomplete cases: '+str(sorted(missing)))
    audit_completed_weekly_cases(STUDY_ROOT)
    export_weekly_market_audits(STUDY_ROOT)
    ex=pd.read_csv(STUDY_ROOT/'tables/extended_validation.csv')
    if not w['PASS'].all(): raise AssertionError('Basic validation failed')
    passcols=[c for c in ex if c.lower()=='pass']
    if not passcols or not ex[passcols[0]].all(): raise AssertionError('Extended validation failed')
    oracle_ns={'__name__':'oracle_export'}
    book=json.loads((ROOT/'upscaledev_weekly_oracle.ipynb').read_text())
    for cell in book['cells']:
        if cell['cell_type']=='code' and 'def export_oracle_comparison_tables' in ''.join(cell['source']):
            exec(''.join(cell['source']),oracle_ns)
    oracle_ns['export_oracle_comparison_tables'](STUDY_ROOT)
    o=pd.read_csv(STUDY_ROOT/'tables/oracle_financial_comparison.csv')
    benchmark=w[w.family.eq('benchmark')].copy()
    benchmark=pd.concat([benchmark,o],ignore_index=True)
    benchmark.to_csv(STUDY_ROOT/'tables/matched_horizon_comparison.csv',index=False)
    def md(frame,columns):
        rows=['| '+' | '.join(columns)+' |','| '+' | '.join(['---']*len(columns))+' |']
        for _,r in frame.iterrows():
            vals=[]
            for c in columns:
                v=r[c]
                vals.append(f'{v:,.2f}' if isinstance(v,(float,np.floating)) else str(v))
            rows.append('| '+' | '.join(vals)+' |')
        return '\n'.join(rows)
    eta=w[w.case_id.isin(['reference','perfect_24h','eta_080','eta_050','eta_080_perfect','eta_050_perfect'])].copy()
    refs=w.set_index('case_id');requested={f:float(refs.loc[c].EV_delivered_kWh) for f,c in [('Persistence','reference'),('Perfect','perfect_24h')]}
    eta['Actual service (%)']=100*eta.EV_delivered_kWh/eta.forecast.map(requested)
    eta['Unserved energy (kWh)']=eta.forecast.map(requested)-eta.EV_delivered_kWh
    eta['Gain per unserved kWh ($/kWh)']=eta['Net revenue improvement vs reference [USD]']/eta['Unserved energy (kWh)'].replace(0,np.nan)
    eta.to_csv(STUDY_ROOT/'tables/service_revenue_tradeoff.csv',index=False)
    lines=[
      '# NTPLL weekly sensitivity results: 1–7 March 2025',
      'Paired Persistence/Perfect results for all four sensitivity families, plus a separate matched-baseline horizon benchmark. All scenarios use passive retail-only PV/building profiles. Paper text and original monthly result folders are unchanged.',
      '## Design and interpretation',
      '- Rolling 24-h MPC, paired Persistence and Perfect using identical scenario settings and synthetic factor profiles; actual CAISO RU/RD attenuation retained. SP/NSP constants 0, 0.2, 0.5, 1; synthetic time-varying profiles have exact daily means 0.2 (three seeds, range 0–0.4) or 0.5 (one seed, range 0–1). These are known scenario inputs, not AGC realizations or forecast-error tests.',
      '- Same February Perfect pre-period EV dispatch seed and March initial PD/NCD thresholds of zero. Sensitivity baselines then evolve with each trajectory. Oracle comparators instead share an explicitly frozen pre-period-derived baseline throughout the week.',
      '- Initial and final BESS SOC are 50%. Base BESS is 250 kW / 332 kWh; power-only, energy-only and joint doubling are tested. No investment cost is included; these are operating outcomes, not sizing recommendations.',
      '- Minimum service means eta times requested energy <= delivered energy <= requested energy, per session. Driver revenue is calculated from delivered energy. Lower service is not a free improvement: unserved energy and potential compensation must be considered.',
      '- Weekly demand charges are increments from zero March peaks, not a prorated seven-day tariff or the completed March bill. Negative total net revenue includes passive building electricity costs and is not standalone EV/BESS profitability.',
      '## Complete weekly financial results (USD; costs signed)',
      md(w,['case_id','Total Revenue','WM Revenue','TOU Cost','PD Cost','NCD Cost','EV Revenue']),
      '## Service/revenue trade-off',
      md(eta,['case_id','eta','Actual service (%)','Unserved energy (kWh)','Net revenue improvement vs reference [USD]','Gain per unserved kWh ($/kWh)']),
      'The last column is the entire operating gain per unserved kWh, before user compensation. It is not a proposed compensation rate.',
      '## Matched-baseline horizon comparison (USD)',
      md(benchmark,['case_id','Total Revenue','WM Revenue','TOU Cost','PD Cost','NCD Cost','EV Revenue']),
      'The standard oracle uses actual-availability DA offer bounds. Persistence DA awards exceed that standard downward envelope by up to 73.216 kW, although its executed RT dispatch is physically feasible. Consequently the retained standard oracle is a full-information benchmark, not a universal feasible-set upper bound for forecast-driven DA awards.',
      md(o,['case_id','objective_incumbent','objective_upper_bound','mip_gap']),
      'A 1% solver tolerance is retained. Feasible oracle revenues are incumbents; objective_upper_bound is the solver certificate, not achieved revenue. Small differences between MPC sensitivities should not be interpreted as a statistically or numerically proven ranking.',
      '## Attenuation interpretation',
      'Let q=p_DA+p_RT+sum(s_x alpha_x c_actual,x), where s_RD=-1 and the other signs are +1. Energy settlement equals dt*[LMP_DA*p_DA+LMP_RT*(p_RT+sum(s_x alpha_x c_actual,x))] = dt*[(LMP_DA-LMP_RT)*p_DA+LMP_RT*q]. Thus at fixed DA energy and physical obligation, explicit attenuation changes can be offset by RT energy adjustments. Feasible capacity sharing and optimal dispatch still change, so invariance is not guaranteed.',
      '## Validation and reproducibility',
      f"{len(w)} completed MPC cases, {int(w.solver_count.sum())} DA/RT optimizations; all accepted within the requested 1% gap, TimeLimit count {int(w.time_limit_count.sum())}. Maximum meter residual {w.meter_error_kW.max():.3e} kW.",
      'Independent checks cover timestamps, signed meter balance, actual energy position, nonnegative AS capacity, EV session service, SOC recursion/bounds/terminal state, hourly DA awards, daily BESS throughput, fixed comparator baseline identity, and revenue reconstruction. AS clock handling in the old oracle loader was corrected; the new oracle matches the source MPC price arrays.',
      '- Source runner: upscaledev_weekly_sensitivity.ipynb. Source oracle: upscaledev_weekly_oracle.ipynb. Main upscaledev_imp_rollingMPC.ipynb only gains the opt-in minimum-service setting; default 1.0 retains full service.',
      '- Figures21–24: plot_weekly_sensitivity() in paper_financial_comparison_plots.ipynb. Top row Persistence, bottom row Perfect, shared scales and colors. Changes use the corresponding forecast-specific full reference; market-participation changes use the corresponding retail-only case. Figure26 retains only the standard oracle. Figures25/27/28 are no longer generated as presentation figures. PNG300dpi copies in Paper_Figures/weekly_sensitivity and Latex/paper3.3/figures.',
      '- tables/paired_sensitivity_paper_table.csv is the compact presentation table. tables/paired_sensitivity_financial_detail.csv and paired_factor/bess/eta/market_financial_detail.csv retain all revenue components by forecast. Small revenue differences at1% solver tolerance are not proven rankings.',
      '- tables/weekly_financial_comparison.csv and daily_financial_comparison.csv: complete financial decomposition. tables/interval_results.csv: every 15-minute dispatch. tables/market_interval_with_prices.csv, market_daily_with_prices.csv and market_weekly_with_prices.csv: energy/four-AS positions, signed obligations, prices and settlements.',
      '- AS energy-revenue columns labeled DA/RT refer to energy associated with DA awards/RT adjustments; both AS deployment-energy components use RT LMP. Capacity payments use their respective DA/RT capacity prices. Daily/week power means are explicitly labeled; interval values are kW.',
      '- Each case retains source/input metadata, solver certificates, March 1 and March 7 six-panel figures and price audit files. No executed notebook copies were created.',
    ]
    (STUDY_ROOT/'RESULTS_README.md').write_text('\n\n'.join(lines)+'\n')
    (STUDY_ROOT/'final_status.json').write_text(json.dumps({'status':'COMPLETE','case_count':len(w),'MPC_optimizations':int(w.solver_count.sum()),'time_limit_count':int(w.time_limit_count.sum()),'all_basic_pass':bool(w.PASS.all()),'all_extended_pass':True,'paper_text_modified':False},indent=2))
    return w
